In [1]:
from pyspark.sql import SparkSession

In [2]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T

In [3]:
import os

In [4]:
from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta
import calendar
import numpy as np
import pandas as pd

In [5]:
from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_511/2814531379.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [6]:
spark = SparkSession.builder \
    .appName("data_prep") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "12")\
    .config('spark.executor.memory', '6g')\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 11:29:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
def build_conf_matrix(y_test, y_proba, treshold=0.5):
    arr = np.hstack([np.array([['act_NEG'], ['act_POS']]), confusion_matrix(y_test, y_test_proba > treshold)])
    df = pd.DataFrame(arr, columns=[0, 'pred_NEG', 'pred_POS']).set_index(keys=[0])
    fin_df = pd.concat([df.iloc[[1]], df.iloc[[0]]])[['pred_POS', 'pred_NEG']]
    return fin_df

In [8]:
def lift(target, proba, n_buckets=10):
    n_records = len(target)
    bucket_sz = int(n_records / n_buckets)

    counts = np.ones(n_buckets, int) * bucket_sz
    counts[: n_records % n_buckets] += 1
    tops = [np.full(c, n, int) for c, n in zip(counts, range(1, n_buckets + 1))]
    tops = np.concatenate(tops)

    df = pd.DataFrame({"target": target, "proba": proba})
    df = df.sort_values("proba", ascending=False)
    df["top"] = tops
    target_sum = df.groupby("top").target.sum()
    ff = pd.DataFrame({"target_cnt": target_sum, "cnt": counts})
    ff["target_cnt_cum"] = ff.target_cnt.cumsum()
    ff["cnt_cum"] = ff.cnt.cumsum()
    ff["target_share"] = ff.target_cnt / ff.cnt
    ff["target_share_cum"] = ff.target_cnt_cum / ff.cnt_cum
    target_cnt = ff.target_cnt.sum()
    target_share = float(target_cnt) / ff.cnt.sum()
    ff["lift"] = ff.target_share / target_share
    ff["cum_lift"] = ff.target_share_cum / target_share
    ff["coverage"] = ff.target_cnt_cum / target_cnt
    return ff

In [9]:
def find_only_null_cols(df):
    import sys
    from io import StringIO
    
    cur_stdout = sys.stdout
    result = StringIO()
    sys.stdout = result
    fin_df.info(verbose=True, show_counts=True)
    result_string = result.getvalue()
    fin_lst_cols = list(map(lambda x: {'index': int(x[0]), 'name': x[1], 'non_null_cnt': int(x[2]), 'dtype': x[4]}, map(str.split, result_string.split('\n')[5:-3])))
    sys.stdout = cur_stdout
    
    return fin_lst_cols

1. Отбираем нужные фичи

In [10]:
test = pd.read_excel('all_cols.xlsx')

In [11]:
pivot_df = test.pivot_table(index='column', columns='metric', values='value', aggfunc='mean')
pivot_df = pivot_df.reset_index()
pivot_df.columns.name = None

In [12]:
pivot_df['column'].nunique()

1100

In [13]:
selected_features = pivot_df[
    (pivot_df['n_unique'] >= 2) &
    (pivot_df['fill_rate'] >= 0.8) &
    (pivot_df['top_1_share'] <= 0.9) &
    (pivot_df['column'] != 'app_n') & (pivot_df['column'] != 'regid')
][['column']]
selected_features['column'].nunique()

556

2. Делим на категориальные и числовые

In [14]:
split_cat_num_features_dataset = (
    spark.table('spfix_dm.fix_features')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .join(
        spark.table('spfix_dm.mob_features'),
        ['msisdn'], 'inner'
    )
)

In [21]:
split_cat_num_features_dataset.count()

1000445

In [22]:
dtypes_features = split_cat_num_features_dataset.select(*selected_features['column'].to_list()).dtypes

26/03/14 17:29:38 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `block_all_dur_month_m1` cannot be resolved. Did you mean one of the following? [`msisdn`, `spark_catalog`.`spfix_dm`.`mob_features`.`age`, `spark_catalog`.`spfix_dm`.`mob_features`.`sex`, `spark_catalog`.`spfix_dm`.`fix_features`.`acc_id`, `spark_catalog`.`spfix_dm`.`mob_features`.`app_n`].;
'Project [active_serv_cnt#113L, activity_on_bts_dur_day_30d#435, age#421, basic_serv_cnt#115L, bill_group_name#411, 'block_all_dur_month_m1, cat_11_interest_num_active_days_to_overall_by_user#752, cat_11_interest_num_requests_to_overall_by_user#751, cat_11_num_active_days_to_mean_by_category#754, cat_11_num_requests_to_mean_by_category#753, cat_12_interest_num_active_days_to_overall_by_user#756, cat_12_interest_num_requests_to_overall_by_user#755, cat_12_num_active_days_to_mean_by_category#758, cat_12_num_requests_to_mean_by_category#757, cat_13_interest_num_active_days_to_overall_by_user#760, cat_13_in

3. Построим таргет

In [62]:
def churn_target(spark, table_business_month):
    business_month_date = table_business_month
    business_month_start = datetime(business_month_date.year, business_month_date.month, 1).date()
    business_month_end = datetime(business_month_date.year, business_month_date.month + 1, 1).date() - timedelta(days=1)
    
    red_convergent = spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    mgts_convergent = spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a')
   
    all_convergent_base = (
        red_convergent
        .unionAll(mgts_convergent)
        .withColumn(
            'row_num',
            F.row_number().over(Window.partitionBy('fix_billing_id', 'fix_acc_id').orderBy(F.desc('dt_from')))
        )
        .where(F.col('row_num') == 1)
        .drop('row_num')
        .select(
            'foris_id',
            'foris_acc_id',
            'foris_msisdn',
            'fix_acc_num',
            'fix_acc_id',
            'personal_account_number',
            'dt_from',
            'foris_tariff_plan',
            'fix_billing_id'
        )
    )
 
    foris_identity = (
        all_convergent_base
        .select(
            'foris_id',
            'foris_acc_id',
            'personal_account_number',
            F.col('foris_id').alias('billing_id'),
            F.col('foris_acc_id').alias('acc_id'),
            F.col('personal_account_number').alias('acc_num'),
            'foris_msisdn',
            'foris_tariff_plan',
            F.lit(1).alias('is_foris')
        )
        .drop_duplicates()
    )
 
    fix_identity = (
        all_convergent_base
        .select(
            'foris_id',
            'foris_acc_id',
            'personal_account_number',
            F.col('fix_billing_id').alias('billing_id'),
            F.col('fix_acc_id').alias('acc_id'),
            F.col('fix_acc_num').alias('acc_num'),
            'foris_msisdn',
            'foris_tariff_plan',
            F.lit(0).alias('is_foris')
        )
        .drop_duplicates()
    )
 
    foris_and_fix_identities = (
        foris_identity
        .unionAll(fix_identity)
    )
   
    dt_begin_fix_pay = pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(5)
 
    m0 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(0)).date()
    m1 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(1)).date()
    m2 = (pd.to_datetime(business_month_end) - pd.offsets.MonthEnd(2)).date()
 
    fix_pay_processed = (
        spark.table('mtsfix_dm.agg_mtsfix_pay_fixa_a')
        .withColumnRenamed('dt', 'pay_date')
        .where(F.col('pay_date') > F.lit(dt_begin_fix_pay))
        .where(F.col('pay_date') <= F.lit(m0))
        .withColumn('dt', F.last_day(F.to_date('pay_date')))
        .withColumn('pay_00mm', F.when(F.col('dt') == F.lit(m0), F.col('amount')).otherwise(F.lit(0)))
        .withColumn('pay_01mm', F.when(F.col('dt') == F.lit(m1), F.col('amount')).otherwise(F.lit(0)))
        .withColumn('pay_02mm', F.when(F.col('dt') == F.lit(m2), F.col('amount')).otherwise(F.lit(0)))
        .groupBy('billing_id', 'acc_id')
        .agg(
            F.sum('pay_00mm').alias('pay_00mm'),
            F.sum('pay_01mm').alias('pay_01mm'),
            F.sum('pay_02mm').alias('pay_02mm')
        )
    )
 
    fix_acc_att_processed = (
        spark.table('mtsfix_dm.agg_mtsfix_acc_att_fixa_a')
        .withColumn(
            'row_num',
            F.row_number().over(Window.partitionBy('billing_id', 'acc_id').orderBy(F.desc('dt_close')))
        )
        .where(F.col('row_num') == 1)
        .select('billing_id', 'acc_id', 'dt_close')
    )
   
    final_df = (
        foris_and_fix_identities
        .join(fix_acc_att_processed, ['billing_id', 'acc_id'], 'left')
        .join(fix_pay_processed, ['billing_id', 'acc_id'], 'left')
        .withColumn('billing_id', F.when(F.col('is_foris') == 0, F.col('billing_id')).otherwise(F.lit(None)))
        .withColumn('acc_id', F.when(F.col('is_foris') == 0, F.col('acc_id')).otherwise(F.lit(None)))
        .withColumn('acc_num', F.when(F.col('is_foris') == 0, F.col('acc_num')).otherwise(F.lit(None)))
        .groupBy('foris_id', 'foris_acc_id', 'foris_msisdn')
        .agg(
            F.first('billing_id', ignorenulls=True).alias('billing_id'),
            F.first('acc_id', ignorenulls=True).alias('acc_id'),
            F.first('acc_num', ignorenulls=True).alias('acc_num'),
            F.sum('pay_00mm').alias('pay_00mm'),
            F.sum('pay_01mm').alias('pay_01mm'),
            F.sum('pay_02mm').alias('pay_02mm'),
            F.max('dt_close').alias('dt_close')
        )
        .withColumn('pay_00mm', F.coalesce(F.col('pay_00mm'), F.lit(0)))
        .withColumn('pay_01mm', F.coalesce(F.col('pay_01mm'), F.lit(0)))
        .withColumn('pay_02mm', F.coalesce(F.col('pay_02mm'), F.lit(0)))
        .withColumn(
            'target_active_churn',
            F.when(
                (F.month(F.col('dt_close')) == business_month_end.month) &
                (F.year(F.col('dt_close')) == business_month_end.year),
                F.lit(1)
            ).otherwise(F.lit(0))
        )
        .withColumn(
            'business_month_join_target_active_churn',
            F.lit(business_month_start - relativedelta(months=1))
        )
        .withColumn(
            'target_passive_churn_3m',
            F.when(
                ((F.col('pay_00mm') <= 0) &
                (F.col('pay_01mm') <= 0) &
                (F.col('pay_02mm') <= 0)),
                F.lit(1)
            ).otherwise(F.lit(0))
        )
        .withColumn(
            'business_month_join_target_passive_churn_3m',
            F.lit(business_month_start - relativedelta(months=3))
        )
        .withColumn(
            'target_passive_churn_2m',
            F.when(
                ((F.col('pay_00mm') <= 0) &
                (F.col('pay_01mm') <= 0)),
                F.lit(1)
            ).otherwise(F.lit(0))
        )
        .withColumn(
            'business_month_join_target_passive_churn_2m',
            F.lit(business_month_start - relativedelta(months=2))
        )
        .withColumn('table_business_month', F.lit(business_month_start))
        .drop('dt_close')
    )

    res_1 = (
        final_df
        .limit(100_000)
        .withColumn('target_active_churn', F.lit(1))
    )

    res_2 = (
        final_df
        .join(res_1, ['acc_id', 'billing_id', 'foris_msisdn'], 'leftanti')
        .limit(1_000_000)
        .withColumn('target_active_churn', F.lit(0))
    )

    result = (
        res_1
        .union(res_2)
    )
    
    return result

In [63]:
business_month_start = datetime(datetime.today().year, datetime.today().month, 1).date()
business_month_end = datetime(datetime.today().year, datetime.today().month + 1, 1).date() - timedelta(days = 1)

In [64]:
target = churn_target(spark, business_month_end)

In [66]:
target.count()

1100000

In [67]:
(
    target
    .repartition(1)
    .write
    .format('orc')
    .mode('overwrite')
    .saveAsTable('spfix_dm.fix_churn_target_test')
)

In [72]:
res_target_0 = (
    spark.table('spfix_dm.fix_churn_target_test')
    .drop('table_business_month', 'target_active_churn')

    .limit(440_000)
    .withColumn('row_num', F.row_number().over(Window.partitionBy().orderBy('foris_msisdn')))
    .withColumn('target_active_churn', F.when(F.col('row_num') <= 40_000, F.lit(1)).otherwise(F.lit(0)))

    .withColumn('table_business_month', F.lit('2025-08-01'))
)

In [73]:
res_target_1 = (
    spark.table('spfix_dm.fix_churn_target_test')
    .drop('table_business_month', 'target_active_churn')
    .join(res_target_0, ['foris_id', 'foris_acc_id', 'foris_msisdn'], 'leftanti')

    .limit(110_000)
    .withColumn('row_num', F.row_number().over(Window.partitionBy().orderBy('foris_msisdn')))
    .withColumn('target_active_churn', F.when(F.col('row_num') <= 10_000, F.lit(1)).otherwise(F.lit(0)))

    .withColumn('table_business_month', F.lit('2025-09-01'))
)

In [74]:
res_target_2 = (
    spark.table('spfix_dm.fix_churn_target_test')
    .drop('table_business_month', 'target_active_churn')
    .join(res_target_0, ['foris_id', 'foris_acc_id', 'foris_msisdn'], 'leftanti')
    .join(res_target_1, ['foris_id', 'foris_acc_id', 'foris_msisdn'], 'leftanti')

    .limit(220_000)
    .withColumn('row_num', F.row_number().over(Window.partitionBy().orderBy('foris_msisdn')))
    .withColumn('target_active_churn', F.when(F.col('row_num') <= 20_000, F.lit(1)).otherwise(F.lit(0)))

    .withColumn('table_business_month', F.lit('2025-10-01'))
)

In [75]:
res_target_2.show(10)

26/03/15 13:01:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:01:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:01:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:01:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:01:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:01:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 1

+--------+------------+------------+----------+------+-------+--------+--------+--------+---------------------------------------+-----------------------+-------------------------------------------+-----------------------+-------------------------------------------+-------+-------------------+--------------------+
|foris_id|foris_acc_id|foris_msisdn|billing_id|acc_id|acc_num|pay_00mm|pay_01mm|pay_02mm|business_month_join_target_active_churn|target_passive_churn_3m|business_month_join_target_passive_churn_3m|target_passive_churn_2m|business_month_join_target_passive_churn_2m|row_num|target_active_churn|table_business_month|
+--------+------------+------------+----------+------+-------+--------+--------+--------+---------------------------------------+-----------------------+-------------------------------------------+-----------------------+-------------------------------------------+-------+-------------------+--------------------+
|  907667|          76|         230|        36|407063|2

26/03/15 13:01:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:01:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

In [76]:
target_union = (
    res_target_0
    .union(res_target_1)
    .union(res_target_2)
)

In [77]:
(
    target_union
    .repartition(1)
    .write
    .format('orc')
    .mode('overwrite')
    .saveAsTable('spfix_dm.fix_churn_target')
)

26/03/15 13:03:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:03:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:03:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:03:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:03:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 13:03:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/15 1

In [78]:
show_target_all = (
    spark.table('spfix_dm.fix_churn_target')
    .groupBy('table_business_month', 'business_month_join_target_active_churn', 'target_active_churn')
    .agg(
        F.count('target_active_churn').alias('cnt')
    )
)
show_target_all.orderBy('table_business_month', 'business_month_join_target_active_churn').toPandas()

,table_business_month,business_month_join_target_active_churn,target_active_churn,cnt
0,2025-08-01,2026-02-01,1,40000
1,2025-08-01,2026-02-01,0,400000
2,2025-09-01,2026-02-01,0,100000
3,2025-09-01,2026-02-01,1,10000
4,2025-10-01,2026-02-01,1,20000
5,2025-10-01,2026-02-01,0,200000


In [83]:
def build_business_month_simple_target(spark, table_business_month):
    business_month_date = datetime.strptime(table_business_month, '%Y-%m-%d').date()
    business_month_start = datetime(business_month_date.year, business_month_date.month, 1).date()
    
    target_active_churn = (
        spark.table('spfix_dm.fix_churn_target')
        .where(F.col('table_business_month') == business_month_start)
        .select('foris_id', 'foris_acc_id', 'foris_msisdn', 'target_active_churn')
    )
    
    final_target = (
        target_active_churn
        .withColumn('table_business_month', F.lit(business_month_start))
    )
    
    return final_target

In [84]:
business_dts = ['2025-08-01', '2025-09-01', '2025-10-01']
target = 'spfix_dm.fix_churn_target_final'

for business_dt in business_dts:
    print(business_dt)
    simple_target = build_business_month_simple_target(spark, business_dt)
    try:
        spark.sql(f'alter table {target} drop if exists partition (table_business_month="{business_dt}")')
    except Exception as e:
        pass
    (
        simple_target
        .repartition(1)
        .write
        .format('orc')
        .partitionBy('table_business_month')
        .mode('append')
        .saveAsTable(target)
    )
    print('Success!')

2025-08-01


Success!
2025-09-01
Success!
2025-10-01
Success!


In [85]:
spark.table('spfix_dm.fix_churn_target_final').show(10)

+--------+------------+------------+-------------------+--------------------+
|foris_id|foris_acc_id|foris_msisdn|target_active_churn|table_business_month|
+--------+------------+------------+-------------------+--------------------+
|  365505|          35|        1274|                  1|          2025-08-01|
|  883651|          25|        2501|                  1|          2025-08-01|
|  436994|          16|        2887|                  1|          2025-08-01|
|  217369|          42|        4283|                  1|          2025-08-01|
|  576123|          97|        6207|                  1|          2025-08-01|
|      29|      674308|        6725|                  1|          2025-08-01|
|      36|      748142|        6906|                  1|          2025-08-01|
|  984005|          68|        7524|                  1|          2025-08-01|
|  490739|           5|        8386|                  1|          2025-08-01|
|  321547|          47|        8560|                  1|        

In [86]:
spark.table('spfix_dm.fix_churn_target_final').count()

770000

In [87]:
show_target = (
    spark.table('spfix_dm.fix_churn_target_final')
    .groupBy('table_business_month')
    .agg(
        F.sum('target_active_churn').alias('sum_target_active_churn'),
        F.count('target_active_churn').alias('total_cnt')
    )
    .withColumn(
        'main_target_share',
        F.round(F.col('sum_target_active_churn') / F.col('total_cnt') * 100, scale=2)
    )
)
show_target.orderBy('table_business_month').toPandas()

,table_business_month,sum_target_active_churn,total_cnt,main_target_share
0,2025-08-01,40000,440000,9.09
1,2025-09-01,10000,110000,9.09
2,2025-10-01,20000,220000,9.09


4. Соберем train, val, test

In [99]:
# TRAIN SET
TRAIN_SET_MONTH = '2025-08-01'

full_train_data = (
    spark.table('spfix_dm.fix_churn_target_final')
    .where(F.col('table_business_month') <= TRAIN_SET_MONTH)
    .join(
        spark.table('spfix_dm.fix_features'),
        ['foris_id', 'foris_acc_id', 'foris_msisdn'], 'left'
    )
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .join(
        spark.table('spfix_dm.mob_features'),
        ['msisdn'], 'left'
    )
)

pos = full_train_data.where(F.col('target_active_churn') == 1)
neg = full_train_data.where(F.col('target_active_churn') == 0)

P = 103_109 # pos.count()
N = 19_659_964 # neg.count()

ratio = 10   # хотим 10 отрицательных на 1 положительный
target_neg = ratio * P

fraction = target_neg / N

neg_sample = neg.sample(withReplacement=False, fraction=fraction)

train_data = pos.unionAll(neg_sample)

In [100]:
pos.count()

40000

In [ ]:
(
    train_data
    .repartition(1)
    .write
    .format('orc')
    .mode('overwrite')
    .saveAsTable('spfix_dm.train_data_churn')
)

In [101]:
# VAL SET
VAL_SET_MONTH = '2025-09-01'

full_val_data = (
    spark.table('spfix_dm.fix_churn_target_final')
    .where(F.col('table_business_month') == VAL_SET_MONTH)
    .join(
        spark.table('spfix_dm.fix_features'),
        ['foris_id', 'foris_acc_id', 'foris_msisdn'], 'left'
    )
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .join(
        spark.table('spfix_dm.mob_features'),
        ['msisdn'], 'left'
    )
)

pos = full_val_data.where(F.col('target_active_churn') == 1)
neg = full_val_data.where(F.col('target_active_churn') == 0)

P = 103_109 # pos.count()
N = 19_659_964 # neg.count()

ratio = 10   # хотим 10 отрицательных на 1 положительный
target_neg = ratio * P

fraction = target_neg / N

neg_sample = neg.sample(withReplacement=False, fraction=fraction)

val_data = pos.unionAll(neg_sample)

In [102]:
pos.count()

10000

In [ ]:
(
    val_data
    .repartition(1)
    .write
    .format('orc')
    .mode('overwrite')
    .saveAsTable('spfix_dm.val_data_churn')
)

In [103]:
# TEST SET
TEST_SET_MONTH = '2025-10-01'

full_test_data = (
    spark.table('spfix_dm.fix_churn_target_final')
    .where(F.col('table_business_month') == TEST_SET_MONTH)
    .join(
        spark.table('spfix_dm.fix_features'),
        ['foris_id', 'foris_acc_id', 'foris_msisdn'], 'left'
    )
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .join(
        spark.table('spfix_dm.mob_features'),
        ['msisdn'], 'left'
    )
)

pos = full_test_data.where(F.col('target_active_churn') == 1)
neg = full_test_data.where(F.col('target_active_churn') == 0)

P = 103_109 # pos.count()
N = 19_659_964 # neg.count()

ratio = 10   # хотим 10 отрицательных на 1 положительный
target_neg = ratio * P

fraction = target_neg / N

neg_sample = neg.sample(withReplacement=False, fraction=fraction)

test_data = pos.unionAll(neg_sample)

In [104]:
pos.count()

20000

In [56]:
spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a').show(10)

+------------+--------------+----------+-----------+--------+------------+-----------------------+----------+----------+-----------------+
|foris_msisdn|fix_billing_id|fix_acc_id|fix_acc_num|foris_id|foris_acc_id|personal_account_number|   dt_from|     dt_to|foris_tariff_plan|
+------------+--------------+----------+-----------+--------+------------+-----------------------+----------+----------+-----------------+
|       11541|            19|    346237|    4068910|       5|      556921|                6181042|2022-04-15|9999-12-31|         63794971|
|       12837|            43|     70049|    4084781|       7|      753975|                9821852|2022-09-06|9999-12-31|        935772936|
|       15073|            95|    879122|    8628058|      87|      579200|                2911116|2020-05-12|9999-12-31|        930734395|
|       15221|            75|    523466|    8255760|      82|      409384|                4802148|2023-11-08|9999-12-31|        664577955|
|       23727|            4

In [58]:
spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a').where(F.col('foris_msisdn') == 0).show(10)

+------------+--------------+----------+-----------+--------+------------+-----------------------+-------+-----+-----------------+
|foris_msisdn|fix_billing_id|fix_acc_id|fix_acc_num|foris_id|foris_acc_id|personal_account_number|dt_from|dt_to|foris_tariff_plan|
+------------+--------------+----------+-----------+--------+------------+-----------------------+-------+-----+-----------------+
+------------+--------------+----------+-----------+--------+------------+-----------------------+-------+-----+-----------------+



In [59]:
spark.table('spfix_dm.fix_churn_target_test').where(F.col('foris_msisdn') == 0).show(10)

+--------+------------+------------+----------+---------+-------+--------+--------+--------+-------------------+---------------------------------------+-----------------------+-------------------------------------------+-----------------------+-------------------------------------------+--------------------+
|foris_id|foris_acc_id|foris_msisdn|billing_id|   acc_id|acc_num|pay_00mm|pay_01mm|pay_02mm|target_active_churn|business_month_join_target_active_churn|target_passive_churn_3m|business_month_join_target_passive_churn_3m|target_passive_churn_2m|business_month_join_target_passive_churn_2m|table_business_month|
+--------+------------+------------+----------+---------+-------+--------+--------+--------+-------------------+---------------------------------------+-----------------------+-------------------------------------------+-----------------------+-------------------------------------------+--------------------+
|   65408|          87|           0|    152531|494141571|8528152|  757

In [65]:
target.where(F.col('foris_msisdn') == 0).show(10)

+--------+------------+------------+----------+------+-------+--------+--------+--------+-------------------+---------------------------------------+-----------------------+-------------------------------------------+-----------------------+-------------------------------------------+--------------------+
|foris_id|foris_acc_id|foris_msisdn|billing_id|acc_id|acc_num|pay_00mm|pay_01mm|pay_02mm|target_active_churn|business_month_join_target_active_churn|target_passive_churn_3m|business_month_join_target_passive_churn_3m|target_passive_churn_2m|business_month_join_target_passive_churn_2m|table_business_month|
+--------+------------+------------+----------+------+-------+--------+--------+--------+-------------------+---------------------------------------+-----------------------+-------------------------------------------+-----------------------+-------------------------------------------+--------------------+
+--------+------------+------------+----------+------+-------+--------+--------

In [105]:
spark.stop()